# CASE 05 — Square 8×8 + 5×5 — Correlation TopLeft

In [1]:
%%writefile case05_sq_5x5_corrTopLeft.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define WIDTH       8
#define HEIGHT      8
#define MASK_WIDTH  5
#define MASK_HEIGHT 5
#define BLOCK_SIZE  8

__global__ void correlationTopLeft(int *dA, int *dMask, int *dC,
                                   int width, int height,
                                   int mWidth, int mHeight)
{
    int col = threadIdx.x + blockIdx.x * blockDim.x;
    int row = threadIdx.y + blockIdx.y * blockDim.y;
    if (row < height && col < width)
    {
        int sum = 0;
        for (int i = 0; i < mHeight; i++)
            for (int j = 0; j < mWidth; j++)
            {
                int r = row + i;        /* top-left */
                int c = col + j;
                if (r < height && c < width)
                    sum += dA[r*width+c] * dMask[i*mWidth+j];
            }
        dC[row*width+col] = sum;
    }
}

void printMatrix(const char *label, int *M, int w, int h)
{
    printf("\n%s:\n", label);
    for (int r = 0; r < h; r++)
    {
        for (int c = 0; c < w; c++)
            printf("%6d", M[r*w+c]);
        printf("\n");
    }
}

int main()
{
    int size     = WIDTH * HEIGHT * sizeof(int);
    int maskSize = MASK_WIDTH * MASK_HEIGHT * sizeof(int);

    int *hA    = (int*) malloc(size);
    int *hMask = (int*) malloc(maskSize);
    int *hC    = (int*) malloc(size);

    srand(time(NULL));
    for (int i = 0; i < WIDTH*HEIGHT; i++)
        hA[i] = rand()%9+1;

    int tempMask[5][5] = {
        {1,1,1,1,1},
        {1,2,2,2,1},
        {1,2,4,2,1},
        {1,2,2,2,1},
        {1,1,1,1,1}
    };
    for (int i = 0; i < MASK_HEIGHT; i++)
        for (int j = 0; j < MASK_WIDTH; j++)
            hMask[i*MASK_WIDTH+j] = tempMask[i][j];

    printMatrix("Input Matrix (8x8)", hA,    WIDTH,      HEIGHT);
    printMatrix("Mask (5x5)",         hMask, MASK_WIDTH, MASK_HEIGHT);

    int *dA, *dMask, *dC;
    cudaMalloc((void**)&dA,    size);
    cudaMalloc((void**)&dMask, maskSize);
    cudaMalloc((void**)&dC,    size);
    cudaMemcpy(dA,    hA,    size,     cudaMemcpyHostToDevice);
    cudaMemcpy(dMask, hMask, maskSize, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, BLOCK_SIZE, 1);
    dim3 DimGrid((int)ceil((float)WIDTH/BLOCK_SIZE),
                 (int)ceil((float)HEIGHT/BLOCK_SIZE), 1);

    cudaEvent_t start, stop; float gpuTime;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);

    correlationTopLeft<<<DimGrid,DimBlock>>>(dA,dMask,dC,
                        WIDTH,HEIGHT,MASK_WIDTH,MASK_HEIGHT);

    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&gpuTime, start, stop);
    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    printMatrix("OUTPUT: Correlation TopLeft (8x8, 5x5)", hC, WIDTH, HEIGHT);
    printf("\nGPU Time: %.4f ms\n", gpuTime);
    printf("Grid: %dx%d  Block: %dx%d\n",
            DimGrid.x,DimGrid.y,DimBlock.x,DimBlock.y);

    cudaFree(dA); cudaFree(dMask); cudaFree(dC);
    free(hA); free(hMask); free(hC);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    return 0;
}

Writing case05_sq_5x5_corrTopLeft.cu


In [2]:
!nvcc -arch=sm_75 case05_sq_5x5_corrTopLeft.cu -o case05_sq_5x5_corrTopLeft

!./case05_sq_5x5_corrTopLeft


Input Matrix (8x8):
     8     4     3     5     4     8     2     1
     8     8     4     9     6     4     6     6
     7     3     7     6     1     1     1     4
     2     2     9     5     2     6     2     7
     9     4     9     1     3     2     1     8
     9     3     5     6     6     9     9     3
     2     7     6     2     5     6     4     4
     7     1     9     6     6     1     3     3

Mask (5x5):
     1     1     1     1     1
     1     2     2     2     1
     1     2     4     2     1
     1     2     2     2     1
     1     1     1     1     1

OUTPUT: Correlation TopLeft (8x8, 5x5):
   196   177   149   132   114    93    55    26
   198   174   151   145   123   106    66    28
   183   164   160   148   133   113    61    26
   179   176   175   173   154   103    59    25
   163   152   156   149   123    87    45    18
   129   124   121   100    86    62    33    10
    67    70    61    50    39    27    17     7
    29    23    25    19    13     